# Model Persistence: Saving and Loading Trained Models

Training is the expensive step; prediction should not require repeating it. This notebook covers three ways to persist a model:

1. **From-scratch numpy model** saved as portable `.npz` (weights) + `.json` (hyper-parameters and metadata), with no pickle involved.
2. **scikit-learn Pipeline** saved with `joblib`.
3. **Verification**: a round-trip must reproduce predictions *exactly*, and loading must fail loudly on a corrupted or mismatched file.

In [ ]:
import json, os, tempfile
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as SkLogReg

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
workdir = tempfile.mkdtemp(prefix='model_persistence_')
print('Artifacts directory:', workdir)

## 1. A from-scratch model that can save itself

The model owns *everything* needed at prediction time: the learned weights **and** the standardisation statistics. Forgetting to persist the scaler's mean/std is the most common persistence bug: the weights load fine but predictions are silently wrong.

In [ ]:
FORMAT_VERSION = 1

class ScratchLogReg:
    def __init__(self, lr=0.1, n_iter=2000, l2=0.01):
        self.lr, self.n_iter, self.l2 = lr, n_iter, l2

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        self.mean_, self.std_ = X.mean(0), X.std(0) + 1e-12
        Z = (X - self.mean_) / self.std_
        self.w_, self.b_ = np.zeros(X.shape[1]), 0.0
        for _ in range(self.n_iter):
            err = self._sigmoid(Z @ self.w_ + self.b_) - y
            self.w_ -= self.lr * (Z.T @ err / len(y) + self.l2 * self.w_)
            self.b_ -= self.lr * err.mean()
        return self

    def predict_proba(self, X):
        return self._sigmoid(((X - self.mean_) / self.std_) @ self.w_ + self.b_)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    # ---- persistence -------------------------------------------------
    def save(self, path):
        meta = {'format_version': FORMAT_VERSION, 'class': type(self).__name__,
                'params': {'lr': self.lr, 'n_iter': self.n_iter, 'l2': self.l2},
                'n_features': int(self.w_.shape[0])}
        np.savez(path + '.npz', w=self.w_, b=np.array(self.b_), mean=self.mean_, std=self.std_)
        with open(path + '.json', 'w') as f:
            json.dump(meta, f, indent=2)

    @classmethod
    def load(cls, path):
        with open(path + '.json') as f:
            meta = json.load(f)
        if meta['format_version'] != FORMAT_VERSION:
            raise ValueError(f"Unsupported format version {meta['format_version']}")
        if meta['class'] != cls.__name__:
            raise ValueError(f"File holds a {meta['class']}, not a {cls.__name__}")
        with np.load(path + '.npz') as data:
            w, b, mean, std = data['w'], float(data['b']), data['mean'], data['std']
        if w.shape[0] != meta['n_features'] or mean.shape != w.shape or std.shape != w.shape:
            raise ValueError('Weight arrays do not match recorded n_features')
        model = cls(**meta['params'])
        model.w_, model.b_, model.mean_, model.std_ = w, b, mean, std
        return model

In [ ]:
model = ScratchLogReg().fit(X_tr, y_tr)
path = os.path.join(workdir, 'scratch_logreg')
model.save(path)

restored = ScratchLogReg.load(path)
print('Original accuracy :', (model.predict(X_te) == y_te).mean())
print('Restored accuracy :', (restored.predict(X_te) == y_te).mean())
print('Probabilities identical:', np.array_equal(model.predict_proba(X_te), restored.predict_proba(X_te)))
print('Files:', sorted(os.listdir(workdir)))

`np.savez` stores float64 arrays bit-for-bit, so the round-trip is exactly lossless. Because the format is plain arrays plus JSON, it is safe to load from a file you did not create, unlike `pickle`/`joblib`, which can execute arbitrary code on load.

## 2. Failure modes: loading must fail loudly

In [ ]:
def expect_error(label, fn):
    try:
        fn()
        print(f'{label}: NO ERROR (bad!)')
    except Exception as e:
        print(f'{label}: {type(e).__name__}: {e}')

# a) wrong format version
bad = os.path.join(workdir, 'bad_version')
model.save(bad)
meta = json.load(open(bad + '.json')); meta['format_version'] = 99
json.dump(meta, open(bad + '.json', 'w'))
expect_error('future version', lambda: ScratchLogReg.load(bad))

# b) weights that disagree with metadata
bad2 = os.path.join(workdir, 'bad_shape')
model.save(bad2)
meta = json.load(open(bad2 + '.json')); meta['n_features'] = 5
json.dump(meta, open(bad2 + '.json', 'w'))
expect_error('shape mismatch', lambda: ScratchLogReg.load(bad2))

# c) missing file
expect_error('missing file', lambda: ScratchLogReg.load(os.path.join(workdir, 'nope')))

## 3. scikit-learn: persist the whole Pipeline

Save the **pipeline**, not just the classifier, so preprocessing travels with the model. `joblib` is efficient for objects holding large numpy arrays. Store the library version alongside it: the artifact is only guaranteed to load under the scikit-learn version that wrote it.

In [ ]:
import sklearn
pipe = Pipeline([('scale', StandardScaler()), ('clf', SkLogReg(max_iter=1000))]).fit(X_tr, y_tr)
pipe_path = os.path.join(workdir, 'pipeline.joblib')
joblib.dump({'pipeline': pipe, 'sklearn_version': sklearn.__version__}, pipe_path, compress=3)

bundle = joblib.load(pipe_path)
loaded = bundle['pipeline']
if bundle['sklearn_version'] != sklearn.__version__:
    print('WARNING: artifact written with sklearn', bundle['sklearn_version'])
print('Predictions identical:', np.array_equal(pipe.predict(X_te), loaded.predict(X_te)))
print('Accuracy:', loaded.score(X_te, y_te))
print('Size on disk (KB): scratch npz = %.1f, joblib pipeline = %.1f' % (
    os.path.getsize(path + '.npz') / 1024, os.path.getsize(pipe_path) / 1024))

## 4. How close are the loaded models to each other?

In [ ]:
p_s = restored.predict_proba(X_te)
p_k = loaded.predict_proba(X_te)[:, 1]
plt.figure(figsize=(5, 5))
plt.scatter(p_k, p_s, s=12, alpha=0.6)
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('sklearn pipeline P(y=1)'); plt.ylabel('restored scratch model P(y=1)')
plt.title('Both loaded models, same test set'); plt.tight_layout(); plt.show()
print('Max |difference|:', np.abs(p_s - p_k).max())

## Takeaways

- Persist **all** state needed for inference (weights *and* preprocessing statistics).
- Prefer a plain-array + JSON format for models you write yourself; use `joblib` only for trusted files.
- Record a format/library version and validate shapes on load so stale or corrupted files raise errors instead of producing wrong predictions.